# nemo Figure 2D: PopGenLM development adaptation

**Runnable cached-output reanalysis, not exact reproduction.** This notebook downloads the immutable, checksum-pinned plotting input, shared metric code and plotting script from the PopGenLM Bench development commit, then recomputes only Panel D. It performs no model inference or training.

Scope: 300 paired *Arabidopsis thaliana* development genes, one validation-selected native nemo reimplementation fit. The target is `log10(median TPM + 0.1)`, a TPM-derived transformed-expression scale, not raw TPM. It is not independent evaluation; the inspected genes remain development data. Exact source-paper 54-of-56 condition selection and architecture/training protocol are unresolved. Panels A-C and original comparators remain unfinished.

Paper: Rockenbach et al. (2026), [doi:10.1093/plphys/kiag473](https://doi.org/10.1093/plphys/kiag473). Measurement source: [NCBI GEO GSE80744](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE80744). The bundled point table is a cited compact figure-data excerpt; see `docs/v0.5_publication_terms.md` for source terms and limitations.

In [ ]:
from pathlib import Path
import hashlib
import tempfile
import urllib.request

REPO = 'tahirali-biomics/popgenlm-bench'
BASE_COMMIT = '628e8d342608477c4f4967bb25d62fe976642459'  # immutable source/data pin
BASE = f'https://raw.githubusercontent.com/{REPO}/{BASE_COMMIT}/'
WORK = Path(tempfile.mkdtemp(prefix='popgenlm-v05-panel-d-'))

ASSETS = {
    'data/benchmarks/v0.5-development/nemo_figure2_panel_d/panel_d_plotting_data.tsv': 'f2f27f2c0a9ab7d8522efbaf2d79186535a43cf4bc84ffbae526b249958fb7e6',
    'data/benchmarks/v0.5-development/nemo_figure2_panel_d/panel_d_metrics.tsv': 'f29f4b1a4085717ab7e423aa0f664c2a50b74cc4e4e03d08d87aa10178b63096',
    'scripts/v0.5/plot_nemo_figure2_partial.py': '938259624665203c3b5e5a197f09244b30d1a8949cb997603e9e309cee4e4841',
    'src/popgenlm/expression.py': 'ee2ac7d4da002c60f3a5b30b142a461899997a3ecafe4f147c1f631da47f926e',
    'requirements-v0.5-panel-d.txt': '766130426c85bc0e395d602d0fd1e16e5cca56a92e1576b56e8e707b61b94255',
}

def fetch_pinned(relative_path, expected_sha256):
    destination = WORK / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(BASE + relative_path, destination)
    digest = hashlib.sha256(destination.read_bytes()).hexdigest()
    if digest != expected_sha256:
        raise RuntimeError(f'checksum mismatch for {relative_path}: {digest}')
    return destination

downloaded = {name: fetch_pinned(name, digest) for name, digest in ASSETS.items()}
print('Pinned commit:', BASE_COMMIT)
print('Downloaded and verified:', len(downloaded), 'files')

In [ ]:
%pip install -q -r {downloaded['requirements-v0.5-panel-d.txt']}
print('Pinned plotting dependencies installed in this notebook kernel.')

In [ ]:
import importlib.util
import sys
import pandas as pd

(WORK / 'src/popgenlm').mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(WORK / 'src'))
module_path = downloaded['scripts/v0.5/plot_nemo_figure2_partial.py']
spec = importlib.util.spec_from_file_location('nemo_figure2_panel_d', module_path)
plotter = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plotter)
output_dir = WORK / 'reanalysis_output'
result = plotter.build_panel(downloaded['data/benchmarks/v0.5-development/nemo_figure2_panel_d/panel_d_plotting_data.tsv'], output_dir)
expected = pd.read_csv(downloaded['data/benchmarks/v0.5-development/nemo_figure2_panel_d/panel_d_metrics.tsv'], sep='\t').iloc[0]
for metric in ['n', 'pearson_r', 'pearson_r_squared', 'predictive_r_squared', 'rmse', 'mae']:
    actual = result['metrics'][metric]
    if metric in ('n',):
        assert int(actual) == int(expected[metric])
    else:
        assert abs(float(actual) - float(expected[metric])) <= 1e-12, (metric, actual, expected[metric])
print(pd.DataFrame([{'metric': k, 'value': result['metrics'][k]} for k in ['n','pearson_r','pearson_r_squared','predictive_r_squared','rmse','mae']]).to_string(index=False))
print('figure2_complete:', result['figure2_complete'])
print('outputs:', sorted(p.name for p in output_dir.iterdir()))

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(output_dir / 'nemo_figure2_panel_d.png')))
print((output_dir / 'caption.md').read_text())

## Interpretation and remaining work

The plot shows observed versus predicted expression for the same 300 development genes, with hexbin density, marginal distributions and an identity line. Signed Pearson *r*, squared Pearson correlation, predictive R², RMSE and MAE are distinct. The single fit does not support training-run uncertainty or superiority claims.

This is **Panel D only**. It does not reproduce the paper’s ten family-fold runs, its complete model suite or source condition selection. Panels A-C, the native/cross-species combinations, original architecture comparators and the confirmatory evaluation remain unfinished. The reserved 600-gene set is not accessed by this notebook.